In [2]:
import numpy as np
import pandas as pd
from collections import Counter

# Load the dataset
df = pd.read_csv("email-Eu-core.txt", sep=" ", header=None, names=["sender", "receiver"])

# Determine the domain size (max node ID + 1)
domain_size = max(df["sender"].max(), df["receiver"].max()) + 1

# Count true frequencies for senders and receivers
true_freq_sender = Counter(df["sender"].tolist())
true_freq_receiver = Counter(df["receiver"].tolist())

true_sender_vec = np.zeros(domain_size)
true_receiver_vec = np.zeros(domain_size)

for k, v in true_freq_sender.items():
    true_sender_vec[k] = v
for k, v in true_freq_receiver.items():
    true_receiver_vec[k] = v

# Combine into one frequency vector for total node communication
true_total_vec = true_sender_vec + true_receiver_vec

# Set epsilon (privacy parameter)
epsilon = 1.0
p = np.exp(epsilon) / (np.exp(epsilon) + domain_size - 1)
q = 1 / (np.exp(epsilon) + domain_size - 1)

# Define DE perturbation function
def direct_encoding_perturb(true_value, domain_size, p, q):
    perturbed = np.full(domain_size, q)
    perturbed[true_value] = p
    return np.random.choice(np.arange(domain_size), p=perturbed)

# Apply DE perturbation for both sender and receiver
perturbed_total = np.zeros(domain_size)

for i in range(len(df)):
    s = df.iloc[i]["sender"]
    r = df.iloc[i]["receiver"]
    s_perturbed = direct_encoding_perturb(s, domain_size, p, q)
    r_perturbed = direct_encoding_perturb(r, domain_size, p, q)
    perturbed_total[s_perturbed] += 1
    perturbed_total[r_perturbed] += 1

# Estimate frequencies (inverse of expected noise)
estimated_total = (perturbed_total - len(df) * 2 * q) / (p - q)
estimated_total = np.clip(estimated_total, 0, None)  # Avoid negatives

# Compute MAE
mae = np.mean(np.abs(estimated_total - true_total_vec))
normalized_mae = mae / np.sum(true_total_vec)

def unary_encoding_perturb(index, domain_size, epsilon):
    unary = np.zeros(domain_size)
    unary[index] = 1

    p = 0.5
    q = 1 / (np.exp(epsilon) + 1)

    for i in range(domain_size):
        rand = np.random.random()
        if unary[i] == 1:
            unary[i] = 1 if rand < p else 0
        else:
            unary[i] = 1 if rand < q else 0
    return unary

# Apply UE perturbation
perturbed_ue = np.zeros(domain_size)
for i in range(len(df)):
    s = df.iloc[i]["sender"]
    r = df.iloc[i]["receiver"]
    s_unary = unary_encoding_perturb(s, domain_size, epsilon)
    r_unary = unary_encoding_perturb(r, domain_size, epsilon)
    perturbed_ue += s_unary + r_unary

# Estimate UE
p = 0.5
q = 1 / (np.exp(epsilon) + 1)
estimated_ue = (perturbed_ue - 2 * len(df) * q) / (p - q)
estimated_ue = np.clip(estimated_ue, 0, None)

# Compute MAE for UE
mae_ue = np.mean(np.abs(estimated_ue - true_total_vec))
normalized_mae_ue = mae_ue / np.sum(true_total_vec)

def optimized_unary_encoding_perturb(index, domain_size, epsilon):
    unary = np.zeros(domain_size)
    unary[index] = 1

    p = 0.5
    q = 1 / (np.exp(epsilon) + 1)

    for i in range(domain_size):
        rand = np.random.random()
        unary[i] = 1 if rand < (p if i == index else q) else 0
    return unary

# Apply OUE perturbation
perturbed_oue = np.zeros(domain_size)
for i in range(len(df)):
    s = df.iloc[i]["sender"]
    r = df.iloc[i]["receiver"]
    s_unary = optimized_unary_encoding_perturb(s, domain_size, epsilon)
    r_unary = optimized_unary_encoding_perturb(r, domain_size, epsilon)
    perturbed_oue += s_unary + r_unary

# Estimate OUE
estimated_oue = (perturbed_oue - 2 * len(df) * q) / (p - q)
estimated_oue = np.clip(estimated_oue, 0, None)

# Compute MAE for OUE
mae_oue = np.mean(np.abs(estimated_oue - true_total_vec))
normalized_mae_oue = mae_oue / np.sum(true_total_vec)

print("Normalized MAE (Direct Encoding, sender+receiver):", normalized_mae)
print("Normalized MAE (Unary Encoding):", normalized_mae_ue)
print("Normalized MAE (Optimized Unary Encoding):", normalized_mae_oue)

Normalized MAE (Direct Encoding, sender+receiver): 0.032909005432526436
Normalized MAE (Unary Encoding): 0.003914543736472182
Normalized MAE (Optimized Unary Encoding): 0.003683956384828774
